# BANKING77 Intent Distribution Check

This notebook checks the intent/category distribution of the BANKING77 dataset from PolyAI-LDN/task-specific-datasets.

Designed for execution in **VS Code with a Colab runtime/plugin**. The first code cell mounts Google Drive and later cells save outputs to Drive.

Data source:
- Repository: https://github.com/PolyAI-LDN/task-specific-datasets
- Train CSV: https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv
- Test CSV: https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv


## 1. Mount Google Drive

Run this first in the VS Code Colab runtime. You will be prompted to authorise access to Google Drive.


In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 2. Imports and configuration

In [ ]:
import os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_rows", 120)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

TRAIN_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/train.csv"
TEST_URL = "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/master/banking_data/test.csv"

PROJECT_DIR = Path("/content/drive/MyDrive/FinDisputeEval")
RUN_ID = globals().get("RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
OUTPUT_DIR = PROJECT_DIR / "outputs" / "data_pipeline" / "banking77_intent_eda" / "eda_v01" / f"run_{RUN_ID}_colab"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Outputs will be saved to: {OUTPUT_DIR}")

## 3. Load BANKING77 train/test CSV files

The original CSV files use two columns:
- `text`: user query
- `category`: intent label


In [ ]:
train_df = pd.read_csv(TRAIN_URL)
test_df = pd.read_csv(TEST_URL)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train columns:", train_df.columns.tolist())
print("Test columns:", test_df.columns.tolist())

display(train_df.head())
display(test_df.head())

## 4. Basic validation

This checks whether the dataset matches the expected BANKING77 structure before analysing the label distribution.


In [ ]:
EXPECTED_COLUMNS = {"text", "category"}
EXPECTED_TRAIN_ROWS = 10003
EXPECTED_TEST_ROWS = 3080
EXPECTED_INTENTS = 77

for split_name, split_df in [("train", train_df), ("test", test_df)]:
    missing_cols = EXPECTED_COLUMNS - set(split_df.columns)
    if missing_cols:
        raise ValueError(f"{split_name} is missing columns: {missing_cols}")

train_df = train_df.copy()
test_df = test_df.copy()
train_df["split"] = "train"
test_df["split"] = "test"

all_df = pd.concat([train_df, test_df], ignore_index=True)

summary = pd.DataFrame({
    "metric": [
        "train_rows", "test_rows", "total_rows", 
        "train_intents", "test_intents", "total_intents",
        "train_null_text", "test_null_text", "train_null_category", "test_null_category",
        "train_duplicate_rows", "test_duplicate_rows"
    ],
    "value": [
        len(train_df), len(test_df), len(all_df),
        train_df["category"].nunique(), test_df["category"].nunique(), all_df["category"].nunique(),
        train_df["text"].isna().sum(), test_df["text"].isna().sum(),
        train_df["category"].isna().sum(), test_df["category"].isna().sum(),
        train_df.duplicated(subset=["text", "category"]).sum(),
        test_df.duplicated(subset=["text", "category"]).sum(),
    ]
})

display(summary)

checks = {
    "train_rows_match_expected": len(train_df) == EXPECTED_TRAIN_ROWS,
    "test_rows_match_expected": len(test_df) == EXPECTED_TEST_ROWS,
    "total_intents_match_expected": all_df["category"].nunique() == EXPECTED_INTENTS,
    "no_null_text": all_df["text"].isna().sum() == 0,
    "no_null_category": all_df["category"].isna().sum() == 0,
}

print("Validation checks:")
for k, v in checks.items():
    print(f"- {k}: {v}")

## 5. Intent distribution table

This table shows how many examples each intent has in train, test, and total.


In [ ]:
intent_counts = (
    all_df
    .groupby(["category", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["train", "test"], fill_value=0)
)

intent_counts["total"] = intent_counts["train"] + intent_counts["test"]
intent_counts["train_pct"] = intent_counts["train"] / intent_counts["train"].sum()
intent_counts["test_pct"] = intent_counts["test"] / intent_counts["test"].sum()
intent_counts["total_pct"] = intent_counts["total"] / intent_counts["total"].sum()
intent_counts["test_train_ratio"] = intent_counts["test"] / intent_counts["train"].replace(0, np.nan)
intent_counts["abs_train_test_pct_gap"] = (intent_counts["train_pct"] - intent_counts["test_pct"]).abs()

intent_counts = intent_counts.sort_values(["total", "category"], ascending=[False, True])

# Cleaner display version
intent_counts_display = intent_counts.copy()
for col in ["train_pct", "test_pct", "total_pct", "abs_train_test_pct_gap"]:
    intent_counts_display[col] = (intent_counts_display[col] * 100).round(3)
intent_counts_display["test_train_ratio"] = intent_counts_display["test_train_ratio"].round(3)

print("Intent distribution sorted by total examples:")
display(intent_counts_display)

intent_counts_display.to_csv(OUTPUT_DIR / "banking77_intent_distribution.csv")
print(f"Saved: {OUTPUT_DIR / 'banking77_intent_distribution.csv'}")

## 6. Distribution summary statistics

Use this to quickly judge whether the dataset is balanced or long-tailed.


In [ ]:
distribution_stats = intent_counts[["train", "test", "total", "test_train_ratio"]].describe().T

distribution_stats["coefficient_of_variation"] = (
    intent_counts[["train", "test", "total"]].std() / intent_counts[["train", "test", "total"]].mean()
).reindex(distribution_stats.index)

display(distribution_stats.round(4))
distribution_stats.to_csv(OUTPUT_DIR / "banking77_distribution_summary_stats.csv")
print(f"Saved: {OUTPUT_DIR / 'banking77_distribution_summary_stats.csv'}")

print("Most frequent intents by total count:")
display(intent_counts_display.head(15))

print("Least frequent intents by total count:")
display(intent_counts_display.tail(15).sort_values(["total", "category"], ascending=[True, True]))

## 7. Train/test coverage and mismatch checks

This cell checks whether any intent appears in only one split, and whether train/test proportions are uneven across intents.


In [ ]:
train_intents = set(train_df["category"].unique())
test_intents = set(test_df["category"].unique())

missing_from_train = sorted(test_intents - train_intents)
missing_from_test = sorted(train_intents - test_intents)

print(f"Intents in train: {len(train_intents)}")
print(f"Intents in test: {len(test_intents)}")
print(f"Missing from train: {missing_from_train}")
print(f"Missing from test: {missing_from_test}")

# Intents with the largest train/test percentage gaps
largest_split_gaps = intent_counts_display.sort_values("abs_train_test_pct_gap", ascending=False).head(15)
print("Largest train/test percentage gaps:")
display(largest_split_gaps)

largest_split_gaps.to_csv(OUTPUT_DIR / "banking77_largest_train_test_distribution_gaps.csv")
print(f"Saved: {OUTPUT_DIR / 'banking77_largest_train_test_distribution_gaps.csv'}")

## 8. Visualise total intent distribution

The full 77-intent distribution is easier to inspect with a horizontal bar chart.


In [ ]:
plot_df = intent_counts.sort_values("total", ascending=True)

fig, ax = plt.subplots(figsize=(12, 18))
ax.barh(plot_df.index, plot_df["total"])
ax.set_title("BANKING77 Total Examples per Intent")
ax.set_xlabel("Number of examples")
ax.set_ylabel("Intent")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

fig_path = OUTPUT_DIR / "banking77_total_intent_distribution.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## 9. Visualise train vs test distribution

This stacked horizontal chart makes it easier to see whether train/test representation is consistent per intent.


In [ ]:
plot_df = intent_counts.sort_values("total", ascending=True)[["train", "test"]]

fig, ax = plt.subplots(figsize=(12, 18))
plot_df.plot(kind="barh", stacked=True, ax=ax)
ax.set_title("BANKING77 Train/Test Examples per Intent")
ax.set_xlabel("Number of examples")
ax.set_ylabel("Intent")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()

fig_path = OUTPUT_DIR / "banking77_train_test_intent_distribution.png"
plt.savefig(fig_path, dpi=200, bbox_inches="tight")
plt.show()
print(f"Saved: {fig_path}")

## 10. Inspect potential long-tail labels

Adjust `LOW_COUNT_THRESHOLD` if you want to define long-tail intents more strictly or loosely.


In [ ]:
LOW_COUNT_THRESHOLD = 120

long_tail = intent_counts_display[intent_counts_display["total"] < LOW_COUNT_THRESHOLD].sort_values(["total", "category"])

print(f"Number of intents with total examples < {LOW_COUNT_THRESHOLD}: {len(long_tail)}")
display(long_tail)

long_tail.to_csv(OUTPUT_DIR / "banking77_long_tail_intents.csv")
print(f"Saved: {OUTPUT_DIR / 'banking77_long_tail_intents.csv'}")

## 11. Example queries per intent

Use this to inspect whether labels look semantically coherent. Change `INTENT_NAME` to any category from the distribution table.


In [ ]:
INTENT_NAME = "card_arrival"  # Change this to inspect another intent
N_EXAMPLES = 10

examples = (
    all_df[all_df["category"] == INTENT_NAME]
    .sample(min(N_EXAMPLES, (all_df["category"] == INTENT_NAME).sum()), random_state=42)
    .reset_index(drop=True)
)

display(examples[["split", "category", "text"]])

## 12. Optional: create a compact Markdown report

This writes a short report to Google Drive summarising the key distribution checks.


In [ ]:
report_path = OUTPUT_DIR / "banking77_intent_distribution_report.md"

min_total = int(intent_counts["total"].min())
max_total = int(intent_counts["total"].max())
mean_total = intent_counts["total"].mean()
median_total = intent_counts["total"].median()
cv_total = intent_counts["total"].std() / intent_counts["total"].mean()

most_common = intent_counts_display.head(10)[["train", "test", "total"]]
least_common = (
    intent_counts_display
    .tail(10)
    .sort_values(["total", "category"], ascending=[True, True])[["train", "test", "total"]]
)

report = []

report.append("# BANKING77 Intent Distribution Report\n\n")

report.append("## Dataset source\n\n")
report.append(f"- Train URL: {TRAIN_URL}\n")
report.append(f"- Test URL: {TEST_URL}\n\n")

report.append("## Basic counts\n\n")
report.append(f"- Train rows: {len(train_df)}\n")
report.append(f"- Test rows: {len(test_df)}\n")
report.append(f"- Total rows: {len(all_df)}\n")
report.append(f"- Total intents: {all_df['category'].nunique()}\n\n")

report.append("## Distribution stats\n\n")
report.append(f"- Min examples per intent: {min_total}\n")
report.append(f"- Max examples per intent: {max_total}\n")
report.append(f"- Mean examples per intent: {mean_total:.2f}\n")
report.append(f"- Median examples per intent: {median_total:.2f}\n")
report.append(f"- Coefficient of variation: {cv_total:.4f}\n\n")

report.append("## Most common intents\n\n")
report.append(most_common.to_markdown())
report.append("\n\n")

report.append("## Least common intents\n\n")
report.append(least_common.to_markdown())
report.append("\n")

report_path.write_text("".join(report), encoding="utf-8")

print(f"Saved: {report_path}")
print(report_path.read_text(encoding="utf-8"))

## 13. Interpretation guide

For benchmark design, check these points:

1. **Intent coverage**: all 77 intents should appear in both train and test.
2. **Class balance**: compare min/median/max counts and coefficient of variation.
3. **Split consistency**: large train/test percentage gaps may indicate uneven representation.
4. **Long-tail intents**: low-count categories are where intent classifiers may be less stable.
5. **Semantic overlap**: inspect examples for similar labels such as refund, card payment, transfer, top-up, and cash withdrawal intents.

For FinDisputeEval-style evaluation, this notebook is useful for understanding the original BANKING77 label distribution. It is not a synthetic dialogue generator.
